# 04 — Preprocessing

Prepares all inputs needed by the four model notebooks.

**Inputs:**
- `data/processed/03b_train.csv` — base + historical features, train
- `data/processed/03b_test.csv`  — base + historical features, test
- `data/processed/03c_instructor.csv` — instructor × course features

**Outputs:**
- `data/processed/04_train.csv`       — 03b + derived regression targets
- `data/processed/04_test.csv`        — 03b + derived regression targets
- `models/scaler.pkl`                 — StandardScaler fit on train only
- `data/processed/04_prof_lookup.csv` — ranked instructor lookup per course

**What this notebook does:**
1. Verify no unexpected nulls in input files
2. Derive regression targets: `target_capacity` and `target_enrollment`
3. Fit StandardScaler on train, apply to test, save scaler
4. Build instructor lookup table (model_prof replacement)

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import sqlite3

DATA_PATH   = Path('../data/processed')
MODELS_PATH = Path('../models')
MODELS_PATH.mkdir(exist_ok=True)
CLEAN_DB = Path('../data/processed/sfu_clean.db')

assert (DATA_PATH / '03b_train.csv').exists(),      'run 03b first'
assert (DATA_PATH / '03b_test.csv').exists(),       'run 03b first'
assert (DATA_PATH / '03c_instructor.csv').exists(), 'run 03c first'
print('ready')

ready


## 1. Load

In [2]:
train = pd.read_csv(DATA_PATH / '03b_train.csv')
test  = pd.read_csv(DATA_PATH / '03b_test.csv')
instr = pd.read_csv(DATA_PATH / '03c_instructor.csv')

print(f'train: {train.shape}')
print(f'test:  {test.shape}')
print(f'instr: {instr.shape}')
print()
print('train columns:')
print(list(train.columns))

train: (47760, 30)
test:  (9552, 30)
instr: (9435, 10)

train columns:
['ml_course_id', 'ml_term_id', 'was_offered', 'year', 'term', 'term_order', 'semester_code', 'is_covid_affected', 'course_level', 'prereq_count', 'has_coreqs', 'units', 'is_grad', 'has_designation', 'is_online', 'is_evening', 'is_burnaby', 'is_surrey', 'is_harbour_ctr', 'is_other_van', 'is_off_campus', 'hist_n_offerings', 'hist_n_sections_total', 'hist_capacity_total', 'hist_enrolled_total', 'hist_n_spring_offerings', 'hist_n_summer_offerings', 'hist_n_fall_offerings', 'hist_fill_rate_std', 'n_terms_since_last_offered']


## 2. Null check

In [3]:
# check train
train_nulls = train.isnull().sum()
print('train nulls:')
print(train_nulls[train_nulls > 0] if train_nulls.any() else 'none')
print()

# check test
test_nulls = test.isnull().sum()
print('test nulls:')
print(test_nulls[test_nulls > 0] if test_nulls.any() else 'none')

train nulls:
none

test nulls:
none


In [4]:
# basic stats on key historical columns
# confirm no negative values in totals
for col in ['hist_n_sections_total','hist_capacity_total','hist_enrolled_total','hist_n_offerings']:
    neg = (train[col] < 0).sum()
    print(f'{col}: min={train[col].min():.0f}  max={train[col].max():.0f}  negatives={neg}')

hist_n_sections_total: min=0  max=131  negatives=0
hist_capacity_total: min=0  max=8640  negatives=0
hist_enrolled_total: min=0  max=7546  negatives=0
hist_n_offerings: min=0  max=14  negatives=0


## 3. Derive regression targets

model_capacity and model_enrollment predict per-section numbers.
We derive these from the stored totals.

- `target_capacity   = hist_capacity_total / hist_n_sections_total`
- `target_enrollment = hist_enrolled_total / hist_n_sections_total`

Rows with hist_n_sections_total = 0 are cold start — no history, no regression target.
These rows are kept in the dataset for model_offered but excluded from capacity/enrollment training.

In [5]:
# load actual section-level data from clean DB
conn = sqlite3.connect(CLEAN_DB)
df   = pd.read_sql('SELECT ml_course_id, ml_term_id, capacity, enrolled FROM offerings', conn)
conn.close()

# actual mean capacity and enrollment per course × term
# (mean because a course can have multiple sections in same term)
actual = (
    df.groupby(['ml_course_id','ml_term_id'])
    .agg(
        target_capacity   = ('capacity', 'mean'),
        target_enrollment = ('enrolled', 'mean')
    )
    .round(1)
    .reset_index()
)

print(f'actual targets: {len(actual):,} course-term pairs')
print()
print('target_capacity stats:')
print(actual['target_capacity'].describe().round(1))
print()
print('target_enrollment stats:')
print(actual['target_enrollment'].describe().round(1))

actual targets: 24,083 course-term pairs

target_capacity stats:
count    24083.0
mean        51.7
std         60.7
min          1.0
25%         15.0
50%         30.0
75%         60.0
max        812.0
Name: target_capacity, dtype: float64

target_enrollment stats:
count    24083.0
mean        37.3
std         50.5
min          1.0
25%          6.0
50%         19.9
75%         44.0
max        567.0
Name: target_enrollment, dtype: float64


In [6]:
# join actual targets onto train and test
train = train.merge(actual, on=['ml_course_id','ml_term_id'], how='left')
test  = test.merge(actual,  on=['ml_course_id','ml_term_id'], how='left')

# rows where was_offered=0 will have NaN targets — course didn't run that term
# rows where was_offered=1 should all have a target
cap_null_train = train['target_capacity'].isna().sum()
cap_null_test  = test['target_capacity'].isna().sum()

print(f'rows with no capacity target — train: {cap_null_train:,}  test: {cap_null_test:,}')
print(f'rows WITH capacity target  — train: {train["target_capacity"].notna().sum():,}  test: {test["target_capacity"].notna().sum():,}')
print()

# sanity check: was_offered=1 rows should all have targets
train_offered_no_target = train[(train['was_offered']==1) & (train['target_capacity'].isna())]
print(f'offered rows with no target (should be 0): {len(train_offered_no_target):,}')

rows with no capacity target — train: 27,983  test: 5,246
rows WITH capacity target  — train: 19,777  test: 4,306

offered rows with no target (should be 0): 0


## 4. StandardScaler

Random Forest and Gradient Boosting do NOT need scaling — tree splits are scale-invariant.
Logistic Regression and KNN DO need scaling — they are distance or weight based.

Rule: fit scaler on train only. Apply same scaler to test. Never fit on test.
Scaler saved to models/scaler.pkl — loaded by 04a and any future notebook that needs it.

In [7]:
# columns to scale — all numeric features, not targets or IDs
# exclude: ml_course_id, ml_term_id (IDs), was_offered (target),
#          target_capacity, target_enrollment (regression targets),
#          binary flags (already 0/1, scaling does nothing useful)

BINARY_COLS = [
    'is_covid_affected','is_grad','has_coreqs','has_designation',
    'is_online','is_evening',
    'is_burnaby','is_surrey','is_harbour_ctr','is_other_van','is_off_campus'
]
ID_COLS      = ['ml_course_id','ml_term_id','semester_code']
TARGET_COLS  = ['was_offered','target_capacity','target_enrollment']
META_COLS    = ['year','term']

SCALE_COLS = [
    c for c in train.columns
    if c not in BINARY_COLS + ID_COLS + TARGET_COLS + META_COLS
    and train[c].dtype in ['int64','float64','int32','float32']
]

print('columns to scale:')
print(SCALE_COLS)

columns to scale:
['term_order', 'course_level', 'prereq_count', 'units', 'hist_n_offerings', 'hist_n_sections_total', 'hist_capacity_total', 'hist_enrolled_total', 'hist_n_spring_offerings', 'hist_n_summer_offerings', 'hist_n_fall_offerings', 'hist_fill_rate_std', 'n_terms_since_last_offered']


In [8]:
# fit on train, transform both
scaler = StandardScaler()
scaler.fit(train[SCALE_COLS])

train_scaled = train.copy()
test_scaled  = test.copy()

train_scaled[SCALE_COLS] = scaler.transform(train[SCALE_COLS])
test_scaled[SCALE_COLS]  = scaler.transform(test[SCALE_COLS])

print('scaler fitted on train')
print(f'mean of course_level after scaling: {train_scaled["course_level"].mean():.4f}  (should be ~0)')
print(f'std  of course_level after scaling: {train_scaled["course_level"].std():.4f}   (should be ~1)')

scaler fitted on train
mean of course_level after scaling: 0.0000  (should be ~0)
std  of course_level after scaling: 1.0000   (should be ~1)


In [9]:
# save scaler
joblib.dump(scaler, MODELS_PATH / 'scaler.pkl')
# also save the list of columns it was fit on
# so model notebooks know exactly which columns to scale
import json
with open(MODELS_PATH / 'scale_cols.json', 'w') as f:
    json.dump(SCALE_COLS, f)

print(f'saved scaler.pkl')
print(f'saved scale_cols.json')

saved scaler.pkl
saved scale_cols.json


## 5. Define feature sets per model

Each model uses a different subset of columns.
Defined once here and saved so model notebooks stay consistent.

In [10]:
# features shared by all three non-prof models
BASE_FEATURES = [
    # IDs (tree models use these as category proxies)
    'ml_course_id',
    'ml_term_id',
    # static course
    'course_level',
    'is_grad',
    'prereq_count',
    'has_coreqs',
    'units',
    'has_designation',
    # term
    'term_order',
    'is_covid_affected',
    # delivery
    'is_online',
    'is_evening',
    'is_burnaby',
    'is_surrey',
    'is_harbour_ctr',
    'is_other_van',
    'is_off_campus',
]

HIST_FEATURES = [
    'hist_n_offerings',
    'hist_n_sections_total',
    'hist_capacity_total',
    'hist_enrolled_total',
    'hist_n_spring_offerings',
    'hist_n_summer_offerings',
    'hist_n_fall_offerings',
    'hist_fill_rate_std',
    'n_terms_since_last_offered',
]

# model_offered uses all features
FEATURES_OFFERED    = BASE_FEATURES + HIST_FEATURES

# model_capacity and model_enrollment use all features
# hist_capacity_total and hist_enrolled_total are still valid features
# (they represent past totals, not the current term's target)
FEATURES_CAPACITY   = BASE_FEATURES + HIST_FEATURES
FEATURES_ENROLLMENT = BASE_FEATURES + HIST_FEATURES

feature_sets = {
    'model_offered':    FEATURES_OFFERED,
    'model_capacity':   FEATURES_CAPACITY,
    'model_enrollment': FEATURES_ENROLLMENT,
}

with open(MODELS_PATH / 'feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)

print('feature sets saved to models/feature_sets.json')
for name, cols in feature_sets.items():
    print(f'  {name}: {len(cols)} features')

feature sets saved to models/feature_sets.json
  model_offered: 26 features
  model_capacity: 26 features
  model_enrollment: 26 features


## 6. Save preprocessed files

Two versions saved:
- `04_train.csv` / `04_test.csv` — unscaled, with targets added. Used by RF and GBM.
- `04_train_scaled.csv` / `04_test_scaled.csv` — scaled, with targets added. Used by LR and KNN.

In [11]:
train.to_csv(DATA_PATH / '04_train.csv', index=False)
test.to_csv( DATA_PATH / '04_test.csv',  index=False)
train_scaled.to_csv(DATA_PATH / '04_train_scaled.csv', index=False)
test_scaled.to_csv( DATA_PATH / '04_test_scaled.csv',  index=False)

print(f'04_train.csv:        {len(train):,} rows  x  {train.shape[1]} cols')
print(f'04_test.csv:         {len(test):,} rows   x  {test.shape[1]} cols')
print(f'04_train_scaled.csv: {len(train_scaled):,} rows  x  {train_scaled.shape[1]} cols')
print(f'04_test_scaled.csv:  {len(test_scaled):,} rows   x  {test_scaled.shape[1]} cols')

04_train.csv:        47,760 rows  x  32 cols
04_test.csv:         9,552 rows   x  32 cols
04_train_scaled.csv: 47,760 rows  x  32 cols
04_test_scaled.csv:  9,552 rows   x  32 cols


## 7. Instructor lookup (model_prof)

No model trained. For a given course, rank instructors by how many times
they have historically taught it. At prediction time, return the top-ranked
instructor for that course.

Also compute a confidence score: times_taught / total_offerings_of_course
This gives the probability that this instructor will teach it, given historical data.

In [12]:
# total historical offerings per course (denominator for confidence)
course_totals = (
    instr
    .groupby('ml_course_id')['instructor_n_times_this_course']
    .sum()
    .reset_index(name='course_total_sections')
)

prof_lookup = instr.merge(course_totals, on='ml_course_id', how='left')

# confidence = fraction of course's sections this instructor taught
prof_lookup['instructor_probability'] = (
    prof_lookup['instructor_n_times_this_course'] /
    prof_lookup['course_total_sections']
).round(3)

# sort: for each course, instructors ranked by probability descending
prof_lookup = prof_lookup.sort_values(
    ['ml_course_id', 'instructor_probability'],
    ascending=[True, False]
).reset_index(drop=True)

print(f'prof_lookup: {len(prof_lookup):,} instructor-course pairs')
prof_lookup.head(5)

prof_lookup: 9,435 instructor-course pairs


,instructor,ml_course_id,instructor_n_times_this_course,instructor_taught_this_course,instructor_n_sections_total,instructor_n_terms_taught,instructor_summer_rate,instructor_fall_rate,instructor_spring_rate,instructor_dept_match,course_total_sections,instructor_probability
0,"Lu, Yi",1,4,1,8,7,0.125,0.500,0.375,1,6,0.667
1,"Ng, Cherie",1,2,1,4,4,0.250,0.500,0.250,1,6,0.333
2,"Jeong, Himchan",2,2,1,3,3,0.000,0.333,0.667,1,4,0.500
3,"Ng, Cherie",2,1,1,4,4,0.250,0.500,0.250,1,4,0.250
4,"Sanders, Barbara",2,1,1,6,4,0.000,0.000,1.000,1,4,0.250


In [13]:
# spot check: CMPT 225 — who taught it most?
# find ml_course_id for CMPT 225 from train
import sqlite3
from pathlib import Path

CLEAN_DB = Path('../data/processed/sfu_clean.db')
conn = sqlite3.connect(CLEAN_DB)
cmpt225 = pd.read_sql(
    "SELECT DISTINCT ml_course_id FROM offerings WHERE dept_code='CMPT' AND course_number='225'",
    conn
)['ml_course_id'].values
conn.close()

print('CMPT 225 — ranked instructors:')
check = prof_lookup[prof_lookup['ml_course_id'].isin(cmpt225)]
check[['instructor','instructor_n_times_this_course','instructor_probability']].head(5)

CMPT 225 — ranked instructors:


,instructor,instructor_n_times_this_course,instructor_probability
1719,"Edgar, John",6,0.207
1720,"Lavergne, Anne",5,0.172
1721,"Mitchell, David",5,0.172
1722,"Donaldson, Toby",2,0.069
1723,"Edgar, John; Imran, Hazra",2,0.069


In [14]:
# distribution check
print('instructor_probability distribution:')
print(prof_lookup['instructor_probability'].describe().round(3))
print()
# how many courses have only one instructor?
single_instr = prof_lookup.groupby('ml_course_id').size()
print(f'courses with only 1 instructor in history: {(single_instr==1).sum():,}')
print(f'courses with 2+ instructors in history:    {(single_instr>1).sum():,}')

instructor_probability distribution:
count    9435.000
mean        0.305
std         0.294
min         0.007
25%         0.083
50%         0.200
75%         0.400
max         1.000
Name: instructor_probability, dtype: float64

courses with only 1 instructor in history: 863
courses with 2+ instructors in history:    2,011


In [15]:
# save
prof_lookup.to_csv(DATA_PATH / '04_prof_lookup.csv', index=False)
print(f'saved 04_prof_lookup.csv: {len(prof_lookup):,} rows  x  {prof_lookup.shape[1]} cols')
print(f'columns: {list(prof_lookup.columns)}')

saved 04_prof_lookup.csv: 9,435 rows  x  12 cols
columns: ['instructor', 'ml_course_id', 'instructor_n_times_this_course', 'instructor_taught_this_course', 'instructor_n_sections_total', 'instructor_n_terms_taught', 'instructor_summer_rate', 'instructor_fall_rate', 'instructor_spring_rate', 'instructor_dept_match', 'course_total_sections', 'instructor_probability']


## 8. Verify all outputs

Quick read-back to confirm all files saved correctly.

In [16]:
files = [
    ('data/processed/04_train.csv',        train.shape),
    ('data/processed/04_test.csv',         test.shape),
    ('data/processed/04_train_scaled.csv', train_scaled.shape),
    ('data/processed/04_test_scaled.csv',  test_scaled.shape),
    ('data/processed/04_prof_lookup.csv',  prof_lookup.shape),
]

for fname, expected_shape in files:
    path = Path('..') / fname
    df   = pd.read_csv(path)
    match = '✓' if df.shape == expected_shape else f'✗ expected {expected_shape}'
    print(f'{fname}: {df.shape}  {match}')

# verify scaler loads
loaded_scaler = joblib.load(MODELS_PATH / 'scaler.pkl')
print(f'scaler.pkl: loaded ok  n_features={loaded_scaler.n_features_in_}')

with open(MODELS_PATH / 'feature_sets.json') as f:
    fs = json.load(f)
print(f'feature_sets.json: loaded ok  {list(fs.keys())}')

data/processed/04_train.csv: (47760, 32)  ✓
data/processed/04_test.csv: (9552, 32)  ✓
data/processed/04_train_scaled.csv: (47760, 32)  ✓
data/processed/04_test_scaled.csv: (9552, 32)  ✓
data/processed/04_prof_lookup.csv: (9435, 12)  ✓
scaler.pkl: loaded ok  n_features=13
feature_sets.json: loaded ok  ['model_offered', 'model_capacity', 'model_enrollment']


## Summary

| Output | Used by |
|--------|--------|
| `04_train.csv` / `04_test.csv` | Random Forest, Gradient Boosting (all 3 tasks) |
| `04_train_scaled.csv` / `04_test_scaled.csv` | Logistic Regression, KNN |
| `models/scaler.pkl` | Load in model notebooks to scale new prediction inputs |
| `models/scale_cols.json` | Which columns the scaler was fit on |
| `models/feature_sets.json` | Feature lists per model |
| `04_prof_lookup.csv` | model_prof — ranked instructor table |
